<a href="https://colab.research.google.com/github/DataSavvyYT/RAG-course/blob/main/02_chunking/chunking_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chunking
Try this website https://chunkviz.up.railway.app/

In [1]:
!pip install -q datasets

In [5]:
!pip install -qU sentence-transformers spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 107.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
from datasets import load_dataset

dataset = load_dataset("jamescalam/ai-arxiv2", split="train")
print(dataset)

Dataset({
    features: ['id', 'title', 'summary', 'source', 'authors', 'categories', 'comment', 'journal_ref', 'primary_category', 'published', 'updated', 'content', 'references'],
    num_rows: 2673
})


In [4]:
content = dataset[3]["content"]
print(content[:500])

# Mamba: Linear-Time Sequence Modeling with Selective State Spaces
# Albert Gu*1 and Tri Dao*2
1Machine Learning Department, Carnegie Mellon University 2Department of Computer Science, Princeton University agu@cs.cmu.edu, tri@tridao.me
# Abstract
Foundation models, now powering most of the exciting applications in deep learning, are almost universally based on the Transformer architecture and its core attention module. Many subquadratic-time architectures such as linear attention, gated convolut


In [6]:
import spacy
import numpy as np
from sentence_transformers import SentenceTransformer, util

In [ ]:
# 1. Load Models
# Using a lightweight MiniLM for speed, or "google/embeddinggemma-2b" for higher accuracy
model = SentenceTransformer('all-MiniLM-L6-v2')
nlp = spacy.load("en_core_web_sm")

In [8]:
def semantic_chunking(text, threshold=0.5):
    # Step A: Split text into sentences using Spacy
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    if len(sentences) <= 1:
        return sentences

    # Step B: Generate embeddings for each sentence
    embeddings = model.encode(sentences, convert_to_tensor=True)

    # Step C: Calculate cosine similarities between adjacent sentences
    # similarities[i] = similarity between sentence[i] and sentence[i+1]
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = util.cos_sim(embeddings[i], embeddings[i+1]).item()
        similarities.append(sim)

    # Step D: Group sentences into chunks based on threshold
    chunks = []
    current_chunk = [sentences[0]]

    for i, sim in enumerate(similarities):
        if sim >= threshold:
            # If sentences are similar, keep them in the same chunk
            current_chunk.append(sentences[i+1])
        else:
            # If similarity drops, start a new chunk
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i+1]]

    # Add the last chunk
    chunks.append(" ".join(current_chunk))
    return chunks


In [12]:
# --- Test Case ---
text_data = """
The Olympic Games are a major international event featuring summer and winter sports.
Thousands of athletes from over 200 nations participate.
The primary goal is to promote peace through sport.
In contrast, deep sea exploration reveals mysteries of the ocean floor.
Giant squids and bioluminescent fish thrive in the high-pressure darkness.
Submersibles allow scientists to map the seabed in high resolution.
"""

In [13]:
final_chunks = semantic_chunking(text_data, threshold=0.45)

In [14]:
for i, chunk in enumerate(final_chunks):
    print(f"--- Chunk {i+1} ---\n{chunk}\n")

--- Chunk 1 ---
The Olympic Games are a major international event featuring summer and winter sports. Thousands of athletes from over 200 nations participate.

--- Chunk 2 ---
The primary goal is to promote peace through sport.

--- Chunk 3 ---
In contrast, deep sea exploration reveals mysteries of the ocean floor.

--- Chunk 4 ---
Giant squids and bioluminescent fish thrive in the high-pressure darkness.

--- Chunk 5 ---
Submersibles allow scientists to map the seabed in high resolution.

